In [0]:
import pyspark
import pyspark.sql.functions as F

In [0]:
catalog_name = "oag"

In [0]:
def read_silver():
    silver_df = spark.table(f"{catalog_name}.silver.silver_table")
    return silver_df

In [0]:
def gold_table_df(silver_df):
    df = silver_df.\
        groupBy(
            "transaction_date",
            "product_name",
            "destination_city")\
            .agg(
            F.round(F.avg("supplier_reliability_score"),2).alias("avg_supplier_reliability_score"),
            F.round(F.sum("ordered_quantity"),2).alias(
                "sum_ordered_quantity"
            ),
            F.round(F.sum("demand_quantity"),2).alias(
                "sum_demand_quantity"
            ),
            F.round(F.sum("available_inventory"),2).alias(
                "sum_available_inventory"
            ),
            F.round(F.avg("unit_price_usd"),2).alias("avg_unit_price_usd"),
            F.round(F.sum("product_cost_usd"),2).alias("sum_product_cost_usd"),
            F.round(F.sum("transportation_cost_usd"),2).alias("sum_transportation_cost_usd"),
            F.round(F.sum("total_cost_usd"),2).alias(
                "sum_total_cost_usd"
            ),
            F.round(F.avg("expected_lead_time_days"),2).alias("avg_expected_lead_time"),
            F.round(F.avg("actual_lead_time_days"),2).alias("avg_actual_lead_time"
            ),
            F.round(F.avg("quality_score"),2).alias("avg_quality_score")
        )
    gold_df = df.withColumn(
                "avg_delay_days",\
                F.round(F.col("avg_actual_lead_time") -F.col("avg_expected_lead_time"), 2))\
                .withColumn("is_delayed",\
                F.when(F.col("avg_delay_days") > 0,1).otherwise(0))\
                .withColumn("is_stockout",F.when(F.col("sum_ordered_quantity") ==F.col("sum_available_inventory"),0).otherwise(1))
    return gold_df


In [0]:
def create_gold_table(df):
    df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog_name}.gold.gold_table")
    return True


In [0]:
df = read_silver()
df.limit(5).display()

In [0]:
gold_df = gold_table_df(df)
gold_df = gold_df.withColumn("timestamp",F.current_timestamp())
gold_df.display()

In [0]:
if create_gold_table(gold_df):
    print("Gold Table Success.")
else:
    print("Gold Table is unsuccessfull")

In [0]:
gold_df = spark.table(f"{catalog_name}.gold.gold_table")

df = gold_df.withColumn("demand_ratio",F.round(F.col("sum_demand_quantity") /F.col("sum_available_inventory"),2))
df.limit(5).display()

In [0]:
df = df.withColumn("remaining_order",\
    F.when(
        F.col("sum_demand_quantity") - F.col("sum_available_inventory") > 0,
        F.round(F.col("sum_demand_quantity") - F.col("sum_available_inventory"),2)
    ).otherwise(0)
)
df.limit(5).show()
